# Data Preparation & Feature Engineering

### Objective

Prepare a clean, consistent, and reproducible dataset that can be used across all subsequent unsupervised learning experiments while preserving clinically meaningful patient similarity.

### - Import Libraries

In [1]:
# Import libraries and modules

# ---------- path setup ----------
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "Notebooks"))
DATA_DIR = PROJECT_ROOT / "Data"


#----- Tools, visualizers -----
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from utils import *

#----- Preprocessing -----
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


### - Load data

In [29]:
#---------------
# Load dataset
#---------------
data = pd.read_csv(DATA_DIR / "Thyroid_Diff.csv")


In [19]:
data.shape

(383, 17)

### 1) Duplicate Handling
A duplicate investigation in `EDA` identified 19 duplicated observations. Manual inspection revealed that most rows shared similar clinicopathologic characteristics but represented distinct patients (e.g., differing in age or other clinical variables). However, two pairs of observations were found to be exact duplicates across all available features, suggesting possible duplicate data entries rather than independent patients. Therefore, only these exact duplicate records were removed, while the remaining observations were retained.

In [30]:
def handle_duplicate(data):
    # Remove only the confirmed duplicate records
    duplicate_indices = [137, 138, 183]  # The indices of samples must be dropped

    data = data.drop(index=duplicate_indices).reset_index(drop=True)

    return data

data_ = handle_duplicate(data)

print("----- Duplicate Handling Done -----")
print(data_.shape)


----- Duplicate Handling Done -----
(380, 17)


### 2) Preprocessong Pipeline

#### Feature Categorization:
| Feature | Data Type | Clinical Role | Recommended Encoding |
|----------|-----------|---------------|----------------------|
| Age | Numerical | Continuous patient characteristic | Standardization |
| Gender | Binary | Demographic | Label Encoding |
| Smoking | Binary | Lifestyle factor | Label Encoding |
| Hx Smoking | Binary | Medical history | Label Encoding |
| Hx Radiothreapy | Binary | Treatment history | Label Encoding |
| Thyroid Function | Nominal | Thyroid status | One-Hot Encoding |
| Physical Examination | Nominal | Clinical examination findings | One-Hot Encoding |
| Adenopathy | Nominal | Lymph node involvement | One-Hot Encoding |
| Pathology | Nominal | Histological subtype | One-Hot Encoding |
| Focality | Binary | Tumor focality | Label Encoding |
| Risk | Ordinal | Clinical risk category | Ordinal Encoding |
| T | Ordinal | Primary tumor stage | Ordinal Encoding |
| N | Ordinal | Lymph node stage | Ordinal Encoding |
| M | Ordinal | Metastasis stage | Ordinal Encoding |
| Stage | Ordinal | Overall cancer stage | Ordinal Encoding |
| Response | Ordinal | Treatment response | Ordinal Encoding *(Feature Set B only)* |
| Recurred | Target / External Reference | Recurrence status | Excluded from training |

#### - Exclude the target and build the sets

In [13]:
def build_feature_sets(data):
    # Feature Set A
    X_A = data.drop(columns=["Response", "Recurred"])

    # Feature Set B
    X_B = data.drop(columns=["Recurred"])

    # External validation variable
    y = data["Recurred"]

    return X_A, X_B, y

X_A, X_B, y = build_feature_sets(data_)
print(f"X_A shape: {X_A.shape}\nX_B shape: {X_B.shape}\ny shape: {y.shape}")

X_A shape: (380, 15)
X_B shape: (380, 16)
y shape: (380,)


#### - The Pipelines

In [ ]:
def prep_pipeline_1():
    """
    This pipeline is for all the unsupervised learning techniques
      except association rule mining
    """

    # Identify columns
    #------------------

    # Continuous numerical features
    num_features = ['Age']

    # Categorical features
    bin_features = ['Gender', 'Smoking', 'Hx Smoking', 'Hx Radiothreapy', 'Focality']
    nom_features = ['Thyroid Function', 'Physical Examination', 'Adenopathy', 'Pathology']
    ord_features_A = ['Risk', 'T', 'N', 'M', 'Stage']
    ord_features_B = ['Risk', 'T', 'N', 'M', 'Stage', 'Response']


    # Continuos numerical Pipelines
    #------------------------------

    # Continuous numerical Pipeline 
    num_Pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])


    # Categorical Pipelines
    #------------------------------

    # Binary features Pipeline
    bin_Pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe_encoder", OneHotEncoder())
    ])

    # Nominal features Pipeline
    nom_Pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe_encoder", OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
    ])

    # Ordinal features Pipeline
    ord_Pipeline = Pipeline(([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("oe_encoder", OrdinalEncoder())
    ]))


    # Combine all Pipelines to build the full preprocessor
    #------------------------------------------------------

    preprocessor_A = ColumnTransformer([
        ("con_num", num_Pipeline, num_features),
        ("bin", bin_Pipeline, bin_features),
        ("nom", nom_Pipeline, nom_features),
        ("ord", ord_Pipeline, ord_features_A)
    ])

    preprocessor_B = ColumnTransformer([
        ("con_num", num_Pipeline, num_features),
        ("bin", bin_Pipeline, bin_features),
        ("nom", nom_Pipeline, nom_features),
        ("ord", ord_Pipeline, ord_features_B)
    ])

    return preprocessor_A, preprocessor_B


preprocessor_A, preprocessor_B = prep_pipeline_1()

print("-----Preprocessing with pipeline done-----", end="\n\n")


-----Preprocessing with pipeline done-----



In [31]:
# discretize/bin numeric features -> Age
def discretize_age(df):
    df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[0,30,45,60,100],
    labels=[
        "Young",
        "Middle",
        "Senior",
        "Elder"
        ]
    )

    df.drop(columns="Age", inplace=True)    
    
    return df

# Apply discretization on Age
bined_df = discretize_age(data_)

bined_df.head()


,Gender,Smoking,Hx Smoking,Hx Radiothreapy,Thyroid Function,Physical Examination,Adenopathy,Pathology,Focality,Risk,T,N,M,Stage,Response,Recurred,AgeGroup
0,F,No,No,No,Euthyroid,Single nodular goiter-left,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Indeterminate,No,Young
1,F,No,Yes,No,Euthyroid,Multinodular goiter,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Excellent,No,Middle
2,F,No,No,No,Euthyroid,Single nodular goiter-right,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Excellent,No,Young
3,F,No,No,No,Euthyroid,Single nodular goiter-right,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Excellent,No,Elder
4,F,No,No,No,Euthyroid,Multinodular goiter,No,Micropapillary,Multi-Focal,Low,T1a,N0,M0,I,Excellent,No,Elder


In [ ]:
def prep_pipeline_2():
    """
    This pipeline is only for association rule mining
      technique of unsupervised learning
    """

    # Identify columns
    #------------------

    # Categorical features
    bin_features = ['Gender', 'Smoking', 'Hx Smoking', 'Hx Radiothreapy', 'Focality', 'Recurred']
    nom_features = ['Thyroid Function', 'Physical Examination', 'Adenopathy',
                     'Pathology', 'Risk', 'T', 'N', 'M', 'Stage', 'Response', 'AgeGroup']


    # Categorical Pipelines
    #------------------------------

    # Binary/nominal features Pipeline
    bin_nom_Pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe_encoder", OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
    ])

    # Combine all Pipelines to build the full preprocessor
    #------------------------------------------------------

    preprocessor = ColumnTransformer([
        ("bin", bin_nom_Pipeline, bin_features),
        ("nom", bin_nom_Pipeline, nom_features)
    ])

    return preprocessor


t_action_preprocessor = prep_pipeline_2()

print("-----Preprocessing with pipeline done-----", end="\n\n")


-----Preprocessing with pipeline done-----



#### - Fit & Transform

In [34]:
# Apply preprocessing

# Feature Set A
X_A_processed = preprocessor_A.fit_transform(X_A)

# Feature Set B
X_B_processed = preprocessor_B.fit_transform(X_B)

# External validation target
y = LabelEncoder().fit_transform(y)

# Transaction dataset
transaction_ds = t_action_preprocessor.fit_transform(bined_df)


print("----- Applied -----")


----- Applied -----


#### - Convert to Dataframe

In [35]:
# Convert to DataFrames
X_A_processed = pd.DataFrame(
    X_A_processed,
    columns=preprocessor_A.get_feature_names_out()
)

X_B_processed = pd.DataFrame(
    X_B_processed,
    columns=preprocessor_B.get_feature_names_out()
)


transaction_ds = pd.DataFrame(
    transaction_ds,
    columns=t_action_preprocessor.get_feature_names_out()
)


print(f"X_A_processed shape: {X_A_processed.shape}\nX_B_processed shape: {X_B_processed.shape}\ntransaction_ds shape: {transaction_ds.shape}")


X_A_processed shape: (380, 36)
X_B_processed shape: (380, 37)
transaction_ds shape: (380, 60)


#### - Inspect the Preprocessed data

In [18]:
X_A_processed

,con_num__Age,bin__Gender_F,bin__Gender_M,bin__Smoking_No,bin__Smoking_Yes,bin__Hx Smoking_No,bin__Hx Smoking_Yes,bin__Hx Radiothreapy_No,bin__Hx Radiothreapy_Yes,bin__Focality_Multi-Focal,...,nom__Adenopathy_Right,nom__Pathology_Follicular,nom__Pathology_Hurthel cell,nom__Pathology_Micropapillary,nom__Pathology_Papillary,ord__Risk,ord__T,ord__N,ord__M,ord__Stage
0,-0.923305,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0
1,-0.460609,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0
2,-0.725007,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0
3,1.390176,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0
4,1.390176,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,2.051171,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,6.0,2.0,1.0,4.0
376,2.646066,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,6.0,2.0,1.0,4.0
377,2.051171,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,6.0,2.0,1.0,4.0
378,1.324077,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,6.0,2.0,0.0,3.0


In [19]:
X_B_processed

,con_num__Age,bin__Gender_F,bin__Gender_M,bin__Smoking_No,bin__Smoking_Yes,bin__Hx Smoking_No,bin__Hx Smoking_Yes,bin__Hx Radiothreapy_No,bin__Hx Radiothreapy_Yes,bin__Focality_Multi-Focal,...,nom__Pathology_Follicular,nom__Pathology_Hurthel cell,nom__Pathology_Micropapillary,nom__Pathology_Papillary,ord__Risk,ord__T,ord__N,ord__M,ord__Stage,ord__Response
0,-0.923305,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,2.0
1,-0.460609,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,1.0
2,-0.725007,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,1.0
3,1.390176,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,1.0
4,1.390176,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,2.051171,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,6.0,2.0,1.0,4.0,0.0
376,2.646066,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,1.0,0.0,6.0,2.0,1.0,4.0,3.0
377,2.051171,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,6.0,2.0,1.0,4.0,3.0
378,1.324077,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,...,0.0,1.0,0.0,0.0,0.0,6.0,2.0,0.0,3.0,3.0


In [36]:
transaction_ds

,bin__Gender_F,bin__Gender_M,bin__Smoking_No,bin__Smoking_Yes,bin__Hx Smoking_No,bin__Hx Smoking_Yes,bin__Hx Radiothreapy_No,bin__Hx Radiothreapy_Yes,bin__Focality_Multi-Focal,bin__Focality_Uni-Focal,...,nom__Stage_IVA,nom__Stage_IVB,nom__Response_Biochemical Incomplete,nom__Response_Excellent,nom__Response_Indeterminate,nom__Response_Structural Incomplete,nom__AgeGroup_Elder,nom__AgeGroup_Middle,nom__AgeGroup_Senior,nom__AgeGroup_Young
0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
3,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
4,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
376,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
377,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
378,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0


#### - Save the sets

In [37]:
X_A_processed.to_csv(DATA_DIR / "feature_set_A.csv", index=False)
X_B_processed.to_csv(DATA_DIR / "feature_set_B.csv", index=False)
pd.DataFrame({"Recurred": y}).to_csv(DATA_DIR / "recurred.csv", index=False)

transaction_ds.to_csv(DATA_DIR / "feature_association_transactions.csv", index=False)

print("----- Sets Saved -----")

----- Sets Saved -----


### Preprocessing Decisions Summary:
| Preprocessing Step | Decision | Justification |
|--------------------|----------|---------------|
| Missing Values | No action required | No missing values were detected. |
| Duplicate Handling | Remove only exact duplicate records | Most duplicated observations represented different patients with similar characteristics, while only exact duplicate entries were removed. |
| Binary Features | Label Encoding | Binary variables naturally map to 0 and 1 without introducing artificial ordering. |
| Nominal Features | One-Hot Encoding | Prevents introducing false ordinal relationships between categories. |
| Ordinal Features | Ordinal Encoding | Preserves clinically meaningful ordering (e.g., Risk, TNM staging). |
| Numerical Features | Standardization | Standardize Age to prevent it from dominating distance-based algorithms. |
| Feature Engineering | None (Baseline) | Preserve the original clinicopathologic characteristics for the initial analysis. |
| Feature Set A | Baseline clinicopathologic features | Represents information available at diagnosis. |
| Feature Set B | Baseline + Response | Evaluates the influence of post-treatment information on clustering. |
| Recurred | Excluded from training | Reserved exclusively for external validation and interpretation of discovered clusters. |

### Preprocessing Decisions Summary (Association Rule Mining)

| Preprocessing Step         | Decision                                    | Justification                                                                                               |
| -------------------------- | ------------------------------------------- | ----------------------------------------------------------------------------------------------------------- |
| Missing Values             | Most Frequent Imputation                    | Preserves categorical information while ensuring complete transactions.                                     |
| Duplicate Handling         | Remove only exact duplicate records         | Identical records are redundant transactions; clinically similar patients are preserved.                    |
| Binary Features            | One-Hot Encoding                            | Converts all binary variables into transaction items suitable for association rule mining.                  |
| Nominal Features           | One-Hot Encoding                            | Represents each category as an independent item without introducing ordinal relationships.                  |
| Ordinal Features           | One-Hot Encoding                            | Association rule mining treats categories as items rather than ordered values.                              |
| Numerical Features         | Discretization (Binning) + One-Hot Encoding | Converts continuous variables into categorical transaction items that can participate in association rules. |
| Feature Scaling            | Not Applied                                 | Distance-based scaling is unnecessary since association rule mining operates on binary transaction data.    |
| Feature Engineering        | Age discretization                          | Produces clinically interpretable age groups for rule discovery.                                            |
| Transaction Representation | Binary Transaction Matrix                   | Required input format for Apriori and FP-Growth algorithms.                                                 |
| Target Variable            | Included in transactions                    | Recurred is treated as another transaction item, allowing discovery of associations with recurrence.        |



### Feature Set A — Baseline Clinicopathologic Features

- Age
- Gender
- Smoking
- Hx Smoking
- Hx Radiothreapy
- Thyroid Function
- Physical Examination
- Adenopathy
- Pathology
- Focality
- Risk
- T
- N
- M
- Stage

---

### Feature Set B — Baseline + Clinical Outcome

Includes all features in **Feature Set A**, plus:

- Response

---

### External Validation Variable

The following feature is **excluded from model training** and will only be used for post hoc interpretation and external validation:

- Recurred

---

### Transaction Dataset

The Association Rule Mining notebook uses one unified transaction dataset.

Unlike previous notebooks, the dataset is not divided into Feature Set A and Feature Set B, because association rule mining searches for relationships among all available clinical variables simultaneously.

#### The final transaction dataset contains:

- Clinical Features
- Gender
- Smoking
- Hx Smoking
- Hx Radiotherapy
- Thyroid Function
- Physical Examination
- Adenopathy
- Pathology
- Focality
- Risk
- T
- N
- M
- Stage
- Response
- Engineered Feature
- AgeGroup (obtained by discretizing Age into categorical intervals)
- Clinical Outcome
- Recurred